імпортуємо бібліотеки 

In [12]:
import os
import pandas as pd
import urllib.request
from datetime import datetime
import re

Для кожної з адміністративних одиниць України завантажити (urllib) тестові структуровані файли, що містять значення VHI-індексу. При зберіганні файлу, до його імені потрібно додати дату та час завантаження. Передбачити повторні запуски скрипту, реалізувати механізм запобігання повторного довантаження та колізії даних;


In [13]:
path = "noaa_data"
if not os.path.exists(path):
    os.makedirs(path)
    print(f"Папка {path} створена.")

def download_data():
    for i in range(1, 28):
        
        existing_files = [f for f in os.listdir(path) if f.startswith(f"vhi_id_{i}_")]
        if existing_files:
            print(f"Область {i} вже завантажена: {existing_files}")
            continue
            
        
        url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={i}&year1=1981&year2=2024&type=Mean"
        now = datetime.now().strftime("%Y%m%d%H%M%S")
        filename = f"vhi_id_{i}_{now}.csv"
        filepath = os.path.join(path, filename)
        
        try:
            print(f"Завантаження області {i}...")
            urllib.request.urlretrieve(url, filepath)
            print(f"Збережено як {filename}")
        except Exception as e:
            print(f"Помилка при завантаженні області {i}: {e}")

download_data()

Область 1 вже завантажена: ['vhi_id_1_20260311144127.csv']
Область 2 вже завантажена: ['vhi_id_2_20260311144134.csv']
Область 3 вже завантажена: ['vhi_id_3_20260311144135.csv']
Область 4 вже завантажена: ['vhi_id_4_20260311144140.csv']
Область 5 вже завантажена: ['vhi_id_5_20260311144141.csv']
Область 6 вже завантажена: ['vhi_id_6_20260311144142.csv']
Область 7 вже завантажена: ['vhi_id_7_20260311144144.csv']
Область 8 вже завантажена: ['vhi_id_8_20260311144146.csv']
Область 9 вже завантажена: ['vhi_id_9_20260311144147.csv']
Область 10 вже завантажена: ['vhi_id_10_20260311144149.csv']
Область 11 вже завантажена: ['vhi_id_11_20260311144150.csv']
Область 12 вже завантажена: ['vhi_id_12_20260311144151.csv']
Область 13 вже завантажена: ['vhi_id_13_20260311144152.csv']
Область 14 вже завантажена: ['vhi_id_14_20260311144153.csv']
Область 15 вже завантажена: ['vhi_id_15_20260311144154.csv']
Область 16 вже завантажена: ['vhi_id_16_20260311144156.csv']
Область 17 вже завантажена: ['vhi_id_17_20

Зчитати завантажені текстові файли у pandas dataframe. Здійснити data cleaning: прибрати зайві стовпці, заповнити пропуски, видалити зайвий текст тощо. Додати стовпчики з назвою та індексом області

In [19]:
regions_dict = {
    1: "Вінницька", 2: "Волинська", 3: "Дніпропетровська", 4: "Донецька", 5: "Житомирська",
    6: "Закарпатська", 7: "Запорізька", 8: "Івано-Франківська", 9: "Київська", 10: "Кіровоградська",
    11: "Луганська", 12: "Львівська", 13: "Миколаївська", 14: "Одеська", 15: "Полтавська",
    16: "Рівненська", 17: "Сумська", 18: "Тернопільська", 19: "Харківська", 20: "Херсонська",
    21: "Хмельницька", 22: "Черкаська", 23: "Чернівецька", 24: "Чернігівська", 25: "Крим"
}

folder = "noaa_data"
all_frames = []
headers = ['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI', 'empty']

if not os.path.exists(folder):
    print("Folder noaa_data not found")
else:
    for filename in os.listdir(folder):
        if filename.endswith(".csv"):
            try:
               
                match = re.search(r'id_(\d+)_', filename)
                if match:
                    province_id = int(match.group(1))
                else:
                    continue

                path = os.path.join(folder, filename)

                
                df = pd.read_csv(path, header=None, skiprows=2, names=headers,na_values=[-1.00])
                
                if 'empty' in df.columns:
                    df = df.drop(columns=['empty'])

                df = df[pd.to_numeric(df['Year'], errors='coerce').notnull()]
                df['Year'] = df['Year'].astype(int)
                df['Week'] = df['Week'].astype(int)
                df = df.interpolate(method='linear') 
                df = df.dropna(subset=['Year'])

              
                df['Province'] = regions_dict.get(province_id, "Unknown")
                df['Province_ID'] = province_id

                all_frames.append(df)

            except Exception as e:
                print("Error in file", filename, ":", e)

    if all_frames:
        main_df = pd.concat(all_frames, ignore_index=True)
        
      
        main_df['Year'] = pd.to_numeric(main_df['Year'])
        main_df['Week'] = pd.to_numeric(main_df['Week'])
        main_df['VHI'] = pd.to_numeric(main_df['VHI'])
        
        print("Data loaded successfully")
        display(main_df.head())
    else:
        print("No data collected. Check files in noaa_data")

Data loaded successfully


,Year,Week,SMN,SMT,VCI,TCI,VHI,Province,Province_ID
0,1982,2,0.063,261.53,55.89,38.20,47.04,Кіровоградська,10
1,1982,3,0.063,263.45,57.30,32.69,44.99,Кіровоградська,10
2,1982,4,0.061,265.10,53.96,28.62,41.29,Кіровоградська,10
3,1982,5,0.058,266.42,46.87,28.57,37.72,Кіровоградська,10
4,1982,6,0.056,267.47,39.55,30.27,34.91,Кіровоградська,10


Ряд VHI для області за вказаний рік;


In [20]:
def get_vhi_by_year(df, province_id, year):
    filtered_df = df[(df['Province_ID'] == province_id) & (df['Year'] == year)]
    return filtered_df[['Year', 'Week', 'VHI', 'Province']]

vhi_series = get_vhi_by_year(main_df, 1, 2020)
display(vhi_series)


,Year,Week,VHI,Province
24325,2020,1,37.80,Вінницька
24326,2020,2,39.94,Вінницька
24327,2020,3,41.75,Вінницька
24328,2020,4,42.76,Вінницька
24329,2020,5,42.19,Вінницька
24330,2020,6,41.29,Вінницька
24331,2020,7,40.53,Вінницька
24332,2020,8,39.70,Вінницька
24333,2020,9,39.11,Вінницька
24334,2020,10,38.47,Вінницька


Ряд VHI за вказаний діапазон років для вказаних областей;


In [21]:
def get_vhi_multiple_regions(df, province_ids, start_year, end_year):
    filtered_df = df[(df['Province_ID'].isin(province_ids)) & 
                     (df['Year'] >= start_year) & 
                     (df['Year'] <= end_year)]
    return filtered_df[['Year', 'Week', 'VHI', 'Province', 'Province_ID']]

multi_vhi = get_vhi_multiple_regions(main_df, [1, 5, 10], 2015, 2020)
display(multi_vhi)

,Year,Week,VHI,Province,Province_ID
1715,2015,1,48.12,Кіровоградська,10
1716,2015,2,51.31,Кіровоградська,10
1717,2015,3,52.05,Кіровоградська,10
1718,2015,4,50.40,Кіровоградська,10
1719,2015,5,46.84,Кіровоградська,10
...,...,...,...,...,...
51192,2020,48,42.12,Житомирська,5
51193,2020,49,43.01,Житомирська,5
51194,2020,50,43.61,Житомирська,5
51195,2020,51,44.95,Житомирська,5


Пошук екстремумів (min та max) для вказаних областей та років, середнього, медіани;


In [18]:
def get_vhi_statistics(df, province_id, year):
    subset = df[(df['Province_ID'] == province_id) & (df['Year'] == year)]
    
    if subset.empty:
        return None
    
    stats = {
        'Min': subset['VHI'].min(),
        'Max': subset['VHI'].max(),
        'Mean': subset['VHI'].mean(),
        'Median': subset['VHI'].median()
    }
    
    print(f"Stats for ID {province_id} ({year}):")
    for key, value in stats.items():
        print(f"{key}: {value:.2f}")
    
    return stats

stats_result = get_vhi_statistics(main_df, 1, 2020)

Stats for ID 1 (2020):
Min: 29.29
Max: 60.90
Mean: 41.38
Median: 39.82
